In [1]:
import os
print(os.getcwd())

/Users/nchaparla/Documents/Capstone_Project


In [2]:
import os

print("Current location:", os.getcwd())
print("\nFiles in raw folder:")
print(os.listdir("01_data/raw/"))

Current location: /Users/nchaparla/Documents/Capstone_Project

Files in raw folder:
['HIQ_L.xpt', '.DS_Store', 'BPXO_L.xpt', 'HUQ_L.xpt', 'DEMO_L.xpt', 'BMX_L.xpt', 'TCHOL_L.xpt', 'BPQ_L.xpt', 'DIQ_L.xpt', 'GHB_L.xpt']


In [3]:
pip install pandas numpy scipy pyreadstat sqlalchemy duckdb matplotlib seaborn scikit-learn xgboost shap

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import os

RAW_PATH = "01_data/raw/"
PROCESSED_PATH = "01_data/processed/"

# Load all 9 NHANES XPT files
demo    = pd.read_sas(os.path.join(RAW_PATH, "DEMO_L.XPT"),   format="xport", encoding="utf-8")
diq     = pd.read_sas(os.path.join(RAW_PATH, "DIQ_L.XPT"),    format="xport", encoding="utf-8")
bpq     = pd.read_sas(os.path.join(RAW_PATH, "BPQ_L.XPT"),    format="xport", encoding="utf-8")
huq     = pd.read_sas(os.path.join(RAW_PATH, "HUQ_L.XPT"),    format="xport", encoding="utf-8")
hiq     = pd.read_sas(os.path.join(RAW_PATH, "HIQ_L.XPT"),    format="xport", encoding="utf-8")
ghb     = pd.read_sas(os.path.join(RAW_PATH, "GHB_L.XPT"),    format="xport", encoding="utf-8")
tchol   = pd.read_sas(os.path.join(RAW_PATH, "TCHOL_L.XPT"),  format="xport", encoding="utf-8")
bpx     = pd.read_sas(os.path.join(RAW_PATH, "BPXO_L.XPT"),   format="xport", encoding="utf-8")
bmx     = pd.read_sas(os.path.join(RAW_PATH, "BMX_L.XPT"),    format="xport", encoding="utf-8")

# Sanity check
files = {"DEMO":demo, "DIQ":diq, "BPQ":bpq, "HUQ":huq,
         "HIQ":hiq, "GHB":ghb, "TCHOL":tchol, "BPX":bpx, "BMX":bmx}

for name, df in files.items():
    print(f"{name}: {df.shape[0]:,} rows | {df.shape[1]} cols | SEQN: {'SEQN' in df.columns}")

DEMO: 11,933 rows | 27 cols | SEQN: True
DIQ: 11,744 rows | 9 cols | SEQN: True
BPQ: 8,501 rows | 6 cols | SEQN: True
HUQ: 11,933 rows | 6 cols | SEQN: True
HIQ: 11,933 rows | 11 cols | SEQN: True
GHB: 7,199 rows | 3 cols | SEQN: True
TCHOL: 8,068 rows | 4 cols | SEQN: True
BPX: 7,801 rows | 12 cols | SEQN: True
BMX: 8,860 rows | 22 cols | SEQN: True


In [5]:
# Start with DEMO as the spine
df = demo.copy()

for name, file in [("DIQ", diq), ("BPQ", bpq), ("HUQ", huq), ("HIQ", hiq),
                   ("GHB", ghb), ("TCHOL", tchol), ("BPX", bpx), ("BMX", bmx)]:
    before = df.shape[0]
    df = df.merge(file, on="SEQN", how="left")
    after = df.shape[0]
    print(f"After merging {name}: {after:,} rows (change: {after - before})")

print(f"\nFinal merged shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

After merging DIQ: 11,933 rows (change: 0)
After merging BPQ: 11,933 rows (change: 0)
After merging HUQ: 11,933 rows (change: 0)
After merging HIQ: 11,933 rows (change: 0)
After merging GHB: 11,933 rows (change: 0)
After merging TCHOL: 11,933 rows (change: 0)
After merging BPX: 11,933 rows (change: 0)
After merging BMX: 11,933 rows (change: 0)

Final merged shape: 11,933 rows × 92 columns


In [6]:
# apply exclusion inclusion criteria

print(f"Before filters: {df.shape[0]:,} rows")

# 1. Adults aged 20 and older
df = df[df["RIDAGEYR"] >= 20]
print(f"After age filter (>=20): {df.shape[0]:,} rows")

# 2. Exclude pregnant women
df = df[~(df["RIDEXPRG"] == 1)]
print(f"After pregnancy exclusion: {df.shape[0]:,} rows")

# 3. Must have at least one primary lab value
has_lab = (
    df["LBXGH"].notna()    |   # HbA1c
    df["BPXOSY1"].notna()  |   # Systolic BP
    df["LBXTC"].notna()        # Total Cholesterol
)
df = df[has_lab]
print(f"After lab value filter: {df.shape[0]:,} rows")
print(f"\nFinal analytic sample: {df.shape[0]:,} rows")

Before filters: 11,933 rows
After age filter (>=20): 7,809 rows
After pregnancy exclusion: 7,768 rows
After lab value filter: 6,004 rows

Final analytic sample: 6,004 rows


In [7]:
import numpy as np

columns_needed = {
    # Identity & Survey Weights
    "SEQN":      "id",
    "WTMEC2YR":  "survey_weight",
    "SDMVPSU":   "psu",
    "SDMVSTRA":  "strata",

    # Demographics
    "RIDAGEYR":  "age",
    "RIAGENDR":  "gender",
    "RIDRETH3":  "race_ethnicity",
    "DMDEDUC2":  "education",
    "INDFMPIR":  "poverty_income_ratio",

    # Self-Reported Diagnoses
    "DIQ010":    "told_diabetic",
    "BPQ020":    "told_hypertensive",
    "BPQ080":    "told_high_cholesterol",

    # Objective Lab & Exam Measurements
    "LBXGH":     "hba1c",
    "BPXOSY1":   "systolic_bp",
    "BPXODI1":   "diastolic_bp",
    "LBXTC":     "total_cholesterol",
    "BMXBMI":    "bmi",
    "BMXWAIST":  "waist_circumference",

    # Healthcare Utilization Outcomes — CORRECTED for 2021-2023 cycle
    "HUQ010":    "self_rated_health",
    "HUQ030":    "usual_care_site",
    "HUQ055":    "outpatient_visits",    # was HUQ051
    "HUQ042":    "er_visits",            # was HUQ071
    "HUQ090":    "overnight_hosp",       # was HUD080

    # Insurance — CORRECTED for 2021-2023 cycle
    "HIQ011":    "has_insurance",
    "HIQ032A":   "private_insurance",    # was HIQ031A
    "HIQ032B":   "medicare",             # was HIQ031B
    "HIQ032D":   "medicaid",             # was HIQ031D
}

# Keep only columns that exist in df
cols_to_keep = {k: v for k, v in columns_needed.items() if k in df.columns}
missing_cols = [k for k in columns_needed.keys() if k not in df.columns]

df = df[list(cols_to_keep.keys())].rename(columns=cols_to_keep)

print(f"Columns kept: {df.shape[1]}")
print(f"Columns not found in data: {missing_cols}")
print(f"\nColumn list:\n{list(df.columns)}")

Columns kept: 27
Columns not found in data: []

Column list:
['id', 'survey_weight', 'psu', 'strata', 'age', 'gender', 'race_ethnicity', 'education', 'poverty_income_ratio', 'told_diabetic', 'told_hypertensive', 'told_high_cholesterol', 'hba1c', 'systolic_bp', 'diastolic_bp', 'total_cholesterol', 'bmi', 'waist_circumference', 'self_rated_health', 'usual_care_site', 'outpatient_visits', 'er_visits', 'overnight_hosp', 'has_insurance', 'private_insurance', 'medicare', 'medicaid']


In [8]:
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct":   (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values("missing_pct", ascending=False)

print(missing[missing["missing_pct"] > 0].to_string())

                      missing_count  missing_pct
medicaid                       5175        86.19
medicare                       3838        63.92
private_insurance              2783        46.35
poverty_income_ratio            781        13.01
er_visits                       675        11.24
total_cholesterol               542         9.03
waist_circumference             286         4.76
hba1c                           274         4.56
diastolic_bp                    180         3.00
systolic_bp                     180         3.00
bmi                              90         1.50
overnight_hosp                    1         0.02


In [9]:
# ── Median imputation for continuous variables ──
continuous_vars = ["hba1c", "total_cholesterol", "systolic_bp",
                   "diastolic_bp", "bmi", "waist_circumference",
                   "poverty_income_ratio"]

for col in continuous_vars:
    if col in df.columns:
        median_val = df[col].median()
        missing_count = df[col].isna().sum()
        df[col] = df[col].fillna(median_val)
        print(f"{col}: filled {missing_count} NAs with median {median_val:.2f}")

# ── Mode imputation for categorical variables ──
categorical_vars = ["education", "usual_care_site",
                    "has_insurance", "race_ethnicity"]

for col in categorical_vars:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

# ── Drop rows with missing outcomes ──
before = df.shape[0]
df = df.dropna(subset=["er_visits", "overnight_hosp"])
print(f"\nDropped {before - df.shape[0]} rows with missing outcomes")
print(f"Rows remaining: {df.shape[0]:,}")

# ── Recode numeric codes to readable labels ──
df["gender"] = df["gender"].map({1: "Male", 2: "Female"})

race_map = {1: "Mexican American", 2: "Other Hispanic",
            3: "Non-Hispanic White", 4: "Non-Hispanic Black",
            6: "Non-Hispanic Asian", 7: "Other/Multiracial"}
df["race_ethnicity"] = df["race_ethnicity"].map(race_map)

diag_map = {1: 1, 2: 0, 7: np.nan, 9: np.nan}
for col in ["told_diabetic", "told_hypertensive", "told_high_cholesterol"]:
    if col in df.columns:
        df[col] = df[col].map(diag_map)

df["has_insurance"] = df["has_insurance"].map({1: 1, 2: 0})

health_map = {1:"Excellent", 2:"Very Good", 3:"Good", 4:"Fair", 5:"Poor"}
df["self_rated_health"] = df["self_rated_health"].map(health_map)

# ── Correct binary outcome variables ──

# er_visits: 1=None, 2=1 time, 3=2 times, 4=3-4 times, 5=5+ times, 99=unknown
# Recode: 1 → 0 (no visit), 2+ → 1 (had a visit), 99 → NaN
df["er_visits"] = df["er_visits"].replace(99.0, np.nan)
df["er_visit_binary"] = (df["er_visits"] >= 2).astype(int)

# overnight_hosp: 1=Yes, 2=No, 7=Refused, 9=Don't know
# Recode: 1 → 1 (hospitalized), 2 → 0 (not hospitalized), 7/9 → NaN
df["overnight_hosp"] = df["overnight_hosp"].replace({7.0: np.nan, 9.0: np.nan})
df["hosp_binary"] = df["overnight_hosp"].map({1.0: 1, 2.0: 0})

# Drop rows still missing outcomes after recoding
before = df.shape[0]
df = df.dropna(subset=["er_visit_binary", "hosp_binary"])
print(f"Dropped {before - df.shape[0]} rows with unresolvable outcome codes")
print(f"Rows remaining: {df.shape[0]:,}")

print(f"\nER visit rate:        {df['er_visit_binary'].mean()*100:.1f}%")
print(f"Hospitalization rate: {df['hosp_binary'].mean()*100:.1f}%")

hba1c: filled 274 NAs with median 5.50
total_cholesterol: filled 542 NAs with median 185.00
systolic_bp: filled 180 NAs with median 121.00
diastolic_bp: filled 180 NAs with median 75.00
bmi: filled 90 NAs with median 28.50
waist_circumference: filled 286 NAs with median 99.50
poverty_income_ratio: filled 781 NAs with median 2.83

Dropped 676 rows with missing outcomes
Rows remaining: 5,328
Dropped 10 rows with unresolvable outcome codes
Rows remaining: 5,318

ER visit rate:        22.2%
Hospitalization rate: 16.5%


In [10]:
# Check actual column names in HUQ and HIQ files
print("HUQ columns:")
print(huq.columns.tolist())

print("\nHIQ columns:")
print(hiq.columns.tolist())

HUQ columns:
['SEQN', 'HUQ010', 'HUQ030', 'HUQ042', 'HUQ055', 'HUQ090']

HIQ columns:
['SEQN', 'HIQ011', 'HIQ032A', 'HIQ032B', 'HIQ032C', 'HIQ032D', 'HIQ032E', 'HIQ032F', 'HIQ032H', 'HIQ032I', 'HIQ210']


In [11]:
# Check raw value distributions for outcome variables
print("er_visits value counts:")
print(df["er_visits"].value_counts().sort_index())

print("\novernight_hosp value counts:")
print(df["overnight_hosp"].value_counts().sort_index())

er_visits value counts:
er_visits
1.0    4135
2.0     662
3.0     261
4.0     186
5.0      66
6.0       6
Name: count, dtype: int64

overnight_hosp value counts:
overnight_hosp
1.0     875
2.0    4443
Name: count, dtype: int64


In [12]:
print("=" * 50)
print("PHASE 1 COMPLETE — FINAL SUMMARY")
print("=" * 50)
print(f"Total participants:        {df.shape[0]:,}")
print(f"Total variables:           {df.shape[1]}")
print(f"\nAge range:                 {df['age'].min():.0f} – {df['age'].max():.0f}")
print(f"\nGender distribution:\n{df['gender'].value_counts()}")
print(f"\nRace/Ethnicity:\n{df['race_ethnicity'].value_counts()}")
print(f"\nER visit rate:             {df['er_visit_binary'].mean()*100:.1f}%")
print(f"Hospitalization rate:      {df['hosp_binary'].mean()*100:.1f}%")
print(f"\nMissing values remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

PHASE 1 COMPLETE — FINAL SUMMARY
Total participants:        5,318
Total variables:           29

Age range:                 20 – 80

Gender distribution:
gender
Female    3005
Male      2313
Name: count, dtype: int64

Race/Ethnicity:
race_ethnicity
Non-Hispanic White    3242
Non-Hispanic Black     669
Other Hispanic         481
Other/Multiracial      329
Mexican American       312
Non-Hispanic Asian     285
Name: count, dtype: int64

ER visit rate:             22.2%
Hospitalization rate:      16.5%

Missing values remaining:
told_diabetic             208
told_hypertensive           3
told_high_cholesterol      31
self_rated_health           2
er_visits                   2
has_insurance               4
private_insurance        2387
medicare                 3240
medicaid                 4570
dtype: int64


In [13]:
output_path = "01_data/processed/nhanes_analytic_clean.csv"
df.to_csv(output_path, index=False)
print(f"✅ Clean dataset saved: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   Location: {output_path}")

✅ Clean dataset saved: 5,318 rows × 29 columns
   Location: 01_data/processed/nhanes_analytic_clean.csv


In [14]:
# Re-recode gender and race from scratch
# First check what values are currently in those columns
print("Gender unique values:", df["gender"].unique())
print("Race unique values:", df["race_ethnicity"].unique())


Gender unique values: ['Male' 'Female']
Race unique values: ['Non-Hispanic Asian' 'Non-Hispanic White' 'Other Hispanic'
 'Mexican American' 'Non-Hispanic Black' 'Other/Multiracial']


In [15]:
# Check raw diagnosis value distributions before any recoding
print("Raw told_diabetic values:")
print(diq["DIQ010"].value_counts(dropna=False).sort_index())

print("\nRaw told_hypertensive values:")
print(bpq["BPQ020"].value_counts(dropna=False).sort_index())

print("\nRaw told_high_cholesterol values:")
print(bpq["BPQ080"].value_counts(dropna=False).sort_index())

Raw told_diabetic values:
DIQ010
1.0     1081
2.0    10371
3.0      284
9.0        4
NaN        4
Name: count, dtype: int64

Raw told_hypertensive values:
BPQ020
1.0    2969
2.0    5518
7.0       1
9.0      10
NaN       3
Name: count, dtype: int64

Raw told_high_cholesterol values:
BPQ080
1.0    3096
2.0    5348
7.0       1
9.0      53
NaN       3
Name: count, dtype: int64


In [17]:
# Check what the id column is called in df
print("df columns:", df.columns.tolist())
print("\nFirst few values of id column:", df["id"].head())


df columns: ['id', 'survey_weight', 'psu', 'strata', 'age', 'education', 'poverty_income_ratio', 'hba1c', 'systolic_bp', 'diastolic_bp', 'total_cholesterol', 'bmi', 'waist_circumference', 'self_rated_health', 'usual_care_site', 'outpatient_visits', 'er_visits', 'overnight_hosp', 'has_insurance', 'private_insurance', 'medicare', 'medicaid', 'er_visit_binary', 'hosp_binary']

First few values of id column: 0    130378.0
1    130379.0
2    130380.0
8    130386.0
9    130387.0
Name: id, dtype: float64


In [20]:
print(df.columns.tolist())

['id', 'survey_weight', 'psu', 'strata', 'age', 'education', 'poverty_income_ratio', 'hba1c', 'systolic_bp', 'diastolic_bp', 'total_cholesterol', 'bmi', 'waist_circumference', 'self_rated_health', 'usual_care_site', 'outpatient_visits', 'er_visits', 'overnight_hosp', 'has_insurance', 'private_insurance', 'medicare', 'medicaid', 'er_visit_binary', 'hosp_binary']


In [22]:
# ── Fix gender, race, and diagnosis columns ──

# Step 1 — Gender and race from demo
demo_fix = demo[["SEQN", "RIAGENDR", "RIDRETH3"]].copy()
demo_fix["RIAGENDR"] = demo_fix["RIAGENDR"].map({1: "Male", 2: "Female"})
demo_fix["RIDRETH3"] = demo_fix["RIDRETH3"].map({
    1: "Mexican American", 2: "Other Hispanic",
    3: "Non-Hispanic White", 4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian", 7: "Other/Multiracial"
})
demo_fix = demo_fix.rename(columns={
    "SEQN":     "id",
    "RIAGENDR": "gender",
    "RIDRETH3": "race_ethnicity"
})

# Step 2 — Diagnosis columns
diq_fix = diq[["SEQN", "DIQ010"]].copy()
diq_fix["DIQ010"] = diq_fix["DIQ010"].map({1.0: 1, 2.0: 0, 3.0: 0, 7.0: np.nan, 9.0: np.nan})
diq_fix = diq_fix.rename(columns={"SEQN": "id", "DIQ010": "told_diabetic"})

bpq_fix = bpq[["SEQN", "BPQ020", "BPQ080"]].copy()
bpq_fix["BPQ020"] = bpq_fix["BPQ020"].map({1.0: 1, 2.0: 0, 7.0: np.nan, 9.0: np.nan})
bpq_fix["BPQ080"] = bpq_fix["BPQ080"].map({1.0: 1, 2.0: 0, 7.0: np.nan, 9.0: np.nan})
bpq_fix = bpq_fix.rename(columns={
    "SEQN":   "id",
    "BPQ020": "told_hypertensive",
    "BPQ080": "told_high_cholesterol"
})

# Step 3 — Merge directly onto df (no drop needed)
df = df.merge(demo_fix, on="id", how="left")
df = df.merge(diq_fix,  on="id", how="left")
df = df.merge(bpq_fix,  on="id", how="left")

# Step 4 — Verify
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

print("\nMissingness on key columns:")
for col in ["gender", "race_ethnicity", "told_diabetic",
            "told_hypertensive", "told_high_cholesterol"]:
    pct = df[col].isna().mean() * 100
    print(f"  {col}: {pct:.1f}% missing")

print(f"\nGender:\n{df['gender'].value_counts()}")
print(f"\nRace/Ethnicity:\n{df['race_ethnicity'].value_counts()}")
print(f"\nTold diabetic:\n{df['told_diabetic'].value_counts()}")
print(f"\nTold hypertensive:\n{df['told_hypertensive'].value_counts()}")
print(f"\nTold high cholesterol:\n{df['told_high_cholesterol'].value_counts()}")

Rows: 5,318 | Columns: 29

Missingness on key columns:
  gender: 0.0% missing
  race_ethnicity: 0.0% missing
  told_diabetic: 0.0% missing
  told_hypertensive: 0.1% missing
  told_high_cholesterol: 0.6% missing

Gender:
gender
Female    3005
Male      2313
Name: count, dtype: int64

Race/Ethnicity:
race_ethnicity
Non-Hispanic White    3242
Non-Hispanic Black     669
Other Hispanic         481
Other/Multiracial      329
Mexican American       312
Non-Hispanic Asian     285
Name: count, dtype: int64

Told diabetic:
told_diabetic
0.0    4494
1.0     824
Name: count, dtype: int64

Told hypertensive:
told_hypertensive
0.0    3125
1.0    2190
Name: count, dtype: int64

Told high cholesterol:
told_high_cholesterol
0.0    2930
1.0    2357
Name: count, dtype: int64


In [23]:
# Overwrite with corrected clean file
output_path = "01_data/processed/nhanes_analytic_clean.csv"
df.to_csv(output_path, index=False)

print(f"✅ Final clean dataset saved")
print(f"   Rows: {df.shape[0]:,}")
print(f"   Columns: {df.shape[1]}")
print(f"\nFinal column list:")
print(df.columns.tolist())

✅ Final clean dataset saved
   Rows: 5,318
   Columns: 29

Final column list:
['id', 'survey_weight', 'psu', 'strata', 'age', 'education', 'poverty_income_ratio', 'hba1c', 'systolic_bp', 'diastolic_bp', 'total_cholesterol', 'bmi', 'waist_circumference', 'self_rated_health', 'usual_care_site', 'outpatient_visits', 'er_visits', 'overnight_hosp', 'has_insurance', 'private_insurance', 'medicare', 'medicaid', 'er_visit_binary', 'hosp_binary', 'gender', 'race_ethnicity', 'told_diabetic', 'told_hypertensive', 'told_high_cholesterol']
